In [1]:
import pandas as pd
import altair as alt

In [5]:
from ecostyles import EcoStyles
# Create styles instance
styles = EcoStyles()
# Register and enable a theme
styles.register_and_enable_theme(theme_name="article")  # or "article"

In [3]:
measles = pd.read_csv('/Users/sambickel-barlow/RADataHub/RADataHub/ChartOfTheDay/US_measles/mAhEh.csv')

In [11]:
measles.head()

,week_number,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,1,18,0,0,0,0,0,4,0.0,60.0
1,2,21,1,1,0,0,1,8,2.0,361.0
2,3,28,12,4,0,0,2,10,15.0,549.0
3,4,28,17,6,0,0,2,12,38.0,742.0
4,5,28,24,10,0,1,2,17,66.0,922.0


In [13]:
melt_cols = measles.columns[1:]

measles_long = measles.melt(id_vars='week_number', value_vars=melt_cols, var_name='year', value_name='cases')

In [83]:
measles_long_2018_2025 = measles_long[measles_long['year'].isin(['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025'])]
measles_long_2026 = measles_long[(measles_long['year'] == '2026') & (~measles_long['cases'].isna())]

tooltip = [
    alt.Tooltip('year:N', title='Year'),
    alt.Tooltip('week_number:Q', title='Week'),
    alt.Tooltip('cases:Q', title='Cumulative Cases', format=','),
]

# Line chart - 2018-2025 in teal, 2026 in red
lines = alt.Chart(measles_long_2018_2025).mark_line().encode(
    x=alt.X('week_number:Q', scale=alt.Scale(domain=[1, 52]), axis=alt.Axis(title='Week Number', titleFontSize=14, labelFontSize=14, values=[1, 10, 20, 30, 40, 50, 52])),
    y=alt.Y('cases:Q', axis=alt.Axis(labelExpr='(datum.value / 1000) + "K"', labelFontSize=14)),
    detail='year:N',
    color=alt.value('#36b7b4'),
    tooltip=tooltip
)

line_2026 = alt.Chart(measles_long_2026).mark_line().encode(
    x='week_number:Q',
    y='cases:Q',
    color=alt.value('#e6224b'),
    tooltip=tooltip
)

# Create a DataFrame for label positions (last week for each year)
label_positions_2018_2025 = measles_long_2018_2025.loc[measles_long_2018_2025.groupby('year')['week_number'].idxmax()]
label_positions_2026 = measles_long_2026.loc[measles_long_2026.groupby('year')['week_number'].idxmax()]
label_positions = pd.concat([label_positions_2018_2025, label_positions_2026]).copy()

# --- Per-year label offsets (dx = horizontal, dy = vertical) ---
label_offsets = {
    '2018': {'dx': 3, 'dy': -2},
    '2019': {'dx': 3, 'dy': 0},
    '2020': {'dx': 3, 'dy': -3},
    '2021': {'dx': 3, 'dy': -8},
    '2022': {'dx': 3, 'dy': -18},
    '2023': {'dx': 3, 'dy': -17},
    '2024': {'dx': 3, 'dy': -3},
    '2025': {'dx': 3, 'dy': 0},
    '2026': {'dx': 3, 'dy': 0},
}

# Per-year label colours
label_colours = {
    '2026': '#e6224b',
}

# Build one text layer per year so each can have its own dx/dy offset
label_layers = []
for _, row in label_positions.iterrows():
    year = str(row['year'])
    offsets = label_offsets.get(year, {'dx': 3, 'dy': -3})
    colour = label_colours.get(year, '#36b7b4')
    label_layers.append(
        alt.Chart(pd.DataFrame([row])).mark_text(
            align='left', fontSize=14,
            dx=offsets['dx'], dy=offsets['dy']
        ).encode(
            x='week_number:Q',
            y='cases:Q',
            text=alt.Text('year:N'),
            color=alt.value(colour),
        )
    )

labels = alt.layer(*label_layers)

# Source note positioned at the bottom of the chart using pixel offset
source_note = alt.Chart(
    pd.DataFrame({'text': ['Source: Johns Hopkins - Bloomberg School of Public Health']})
).mark_text(align='left', fontSize=12, fontStyle='italic', color='gray').encode(
    x=alt.value(0),
    y=alt.value(450),
    text='text:N'
)

final_chart = (lines + line_2026 + labels + source_note).properties(
    title=alt.TitleParams(
        text='Cumulative Reported Measles Cases in the US by Year',
        fontSize=18,
    ),
    width=600,
    height=400
)

final_chart

alt.LayerChart(...)

In [84]:
# Save to png
final_chart.save('measles_chart.png', scale_factor=2)
# Save to json
final_chart.save('measles_chart.json', scale_factor=2)